# 11-4. OLS: 2025년 승차인원과 2024년 승차규모 통제

셀×노선별 2025년 종일 서울행 광역버스 승차인원을 종속변수로 사용하고, 2024년 승차인원을 일반 통제변수로 포함합니다. 노선 고정효과를 적용하며, 같은 격자에 여러 노선이 포함되는 구조를 고려하여 셀 단위 군집 강건 표준오차를 사용합니다. 모든 연속형 변수는 표준화하지 않은 원단위입니다.

In [1]:
import sys, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
import config as cfg
cfg = importlib.reload(cfg)
cfg.setup_notebook()
ID, ROUTE = '셀 ID', '노선'
print('설정 파일:', Path(cfg.__file__).resolve())


설정 파일: C:\Users\금경훈\Desktop\Gachon\3-2\UROP\github_2026-UROP\gtx_a_bus_analysis\config.py


## 1. 셀×노선 분석자료 구축

In [2]:
grid = cfg.load_model_data()
routes = cfg.load_cell_route_all_day()
grid_cols = [ID, 'walk_min', 'walk_20', 'walk_20_60', 'ic_access', 'pop_20_50', 'buildings']
data = routes.merge(grid[grid_cols], on=ID, how='inner', validate='many_to_one')
numeric = ['y2024', 'y2025', 'walk_min', 'walk_20', 'walk_20_60', 'ic_access', 'competing_share', 'pop_20_50', 'buildings']
for column in numeric:
    data[column] = pd.to_numeric(data[column], errors='coerce')
data = data.dropna(subset=numeric + [ID, ROUTE]).reset_index(drop=True)
data['승차인원_증감'] = data['y2025'] - data['y2024']
print('관측치:', len(data), '| 격자:', data[ID].nunique(), '| 노선:', data[ROUTE].nunique())
display(data.head())


관측치: 220 | 격자: 132 | 노선: 24


,셀 ID,노선,y2024,y2025,competing_share,competing_share_missing,walk_min,walk_20,walk_20_60,ic_access,pop_20_50,buildings,승차인원_증감
0,44,1200,269,164,21.189591,0,51.650000,0.0,1.0,5.071,937.0,18.0,-105
1,76,2200,17,35,0.000000,0,160.710000,0.0,0.0,1.921,33.0,37.0,18
2,559,3100,82,90,0.000000,0,50.225000,0.0,1.0,0.319,0.0,46.0,8
3,619,9710,1,4,100.000000,0,166.555000,0.0,0.0,9.029,0.0,7.0,3
4,765,9709,12,14,25.000000,0,128.416667,0.0,0.0,4.323,139.0,92.0,2


## 2. OLS 추정 함수

In [3]:
LABELS = {
    'y2024':'2024년 종일 승차인원',
    'walk_min':'GTX역 보행시간(분)',
    'walk_20':'GTX 보행 20분 이내',
    'walk_20_60':'GTX 보행 20~60분',
    'ic_access':'IC까지 도로거리(km)',
    'competing_share':'GTX 경쟁 목적지 비율(%p)',
    'pop_20_50':'2024년 20~50대 인구(명)',
    'buildings':'2024년 건축물 수(동)'
}
def fit_ols(name, features):
    d = data.dropna(subset=['y2025'] + features).copy()
    route_fe = pd.get_dummies(d[ROUTE].astype(str), prefix='노선', drop_first=True, dtype=float)
    X = sm.add_constant(pd.concat([d[features].astype(float), route_fe], axis=1), has_constant='add')
    model = sm.OLS(d['y2025'].astype(float), X)
    classical = model.fit()
    clustered = model.fit(cov_type='cluster', cov_kwds={'groups':pd.factorize(d[ID])[0], 'use_correction':True})
    ci = clustered.conf_int().loc[features]
    result = pd.DataFrame({
        '변수':[LABELS[v] for v in features],
        '계수':[clustered.params[v] for v in features],
        '군집 표준오차':[clustered.bse[v] for v in features],
        't값':[clustered.tvalues[v] for v in features],
        'p-value':[clustered.pvalues[v] for v in features],
        '95% CI 하한':[ci.loc[v,0] for v in features],
        '95% CI 상한':[ci.loc[v,1] for v in features]
    })
    info = {
        '모형':name, '표본 수':len(d), '격자 수':d[ID].nunique(), '노선 수':d[ROUTE].nunique(),
        'R²':classical.rsquared, '수정 R²':classical.rsquared_adj,
        '군집강건 F값':float(clustered.fvalue), '군집강건 F p값':float(clustered.f_pvalue),
        'AIC':classical.aic, 'BIC':classical.bic, 'RMSE':float(np.sqrt(np.mean(classical.resid**2)))
    }
    return {'name':name, 'data':d, 'X':X, 'classical':classical, 'fit':clustered, 'result':result, 'info':info}


## 3. 모형 1 — GTX 보행시간 구간 더미

보행시간 60분 초과 지역이 기준집단입니다.

In [4]:
DUMMY_FEATURES = ['y2024', 'walk_20', 'walk_20_60', 'ic_access', 'competing_share', 'pop_20_50', 'buildings']
ols_dummy = fit_ols('OLS_보행시간구간더미', DUMMY_FEATURES)
display(pd.DataFrame([ols_dummy['info']]).round(4))
display(ols_dummy['result'].round(4))


,모형,표본 수,격자 수,노선 수,R²,수정 R²,군집강건 F값,군집강건 F p값,AIC,BIC,RMSE
0,OLS_보행시간구간더미,220,132,24,0.7664,0.7293,93.4178,0.0,2560.9584,2666.1608,70.8463


,변수,계수,군집 표준오차,t값,p-value,95% CI 하한,95% CI 상한
0,2024년 종일 승차인원,0.7506,0.1531,4.9026,0.0000,0.4506,1.0507
1,GTX 보행 20분 이내,-56.2864,31.5896,-1.7818,0.0748,-118.2010,5.6281
2,GTX 보행 20~60분,-9.6186,11.3303,-0.8489,0.3959,-31.8256,12.5884
3,IC까지 도로거리(km),-3.3555,2.3792,-1.4103,0.1584,-8.0187,1.3077
4,GTX 경쟁 목적지 비율(%p),-0.0699,0.0991,-0.7059,0.4803,-0.2641,0.1242
5,2024년 20~50대 인구(명),0.0127,0.0179,0.7099,0.4778,-0.0224,0.0479
6,2024년 건축물 수(동),0.0195,0.1760,0.1105,0.9120,-0.3256,0.3645


## 4. 모형 2 — GTX 보행시간 연속형

In [5]:
CONTINUOUS_FEATURES = ['y2024', 'walk_min', 'ic_access', 'competing_share', 'pop_20_50', 'buildings']
ols_continuous = fit_ols('OLS_보행시간연속형', CONTINUOUS_FEATURES)
display(pd.DataFrame([ols_continuous['info']]).round(4))
display(ols_continuous['result'].round(4))


,모형,표본 수,격자 수,노선 수,R²,수정 R²,군집강건 F값,군집강건 F p값,AIC,BIC,RMSE
0,OLS_보행시간연속형,220,132,24,0.7593,0.7226,168.971,0.0,2565.4824,2667.2912,71.9046


,변수,계수,군집 표준오차,t값,p-value,95% CI 하한,95% CI 상한
0,2024년 종일 승차인원,0.7293,0.1614,4.5175,0.0000,0.4129,1.0456
1,GTX역 보행시간(분),0.1015,0.1714,0.5920,0.5538,-0.2344,0.4373
2,IC까지 도로거리(km),-2.3381,2.5747,-0.9081,0.3638,-7.3844,2.7082
3,GTX 경쟁 목적지 비율(%p),-0.0952,0.1082,-0.8799,0.3789,-0.3071,0.1168
4,2024년 20~50대 인구(명),0.0130,0.0181,0.7187,0.4723,-0.0224,0.0484
5,2024년 건축물 수(동),-0.0606,0.1541,-0.3934,0.6940,-0.3627,0.2414


## 5. 두 OLS 모형 비교

In [6]:
comparison = pd.DataFrame([ols_dummy['info'], ols_continuous['info']])
display(comparison.round(4))
results = []
for model in [ols_dummy, ols_continuous]:
    table = model['result'][['변수', '계수', '군집 표준오차', 't값', 'p-value']].copy()
    table.insert(0, '모형', model['name'])
    results.append(table)
display(pd.concat(results, ignore_index=True).round(4))


,모형,표본 수,격자 수,노선 수,R²,수정 R²,군집강건 F값,군집강건 F p값,AIC,BIC,RMSE
0,OLS_보행시간구간더미,220,132,24,0.7664,0.7293,93.4178,0.0,2560.9584,2666.1608,70.8463
1,OLS_보행시간연속형,220,132,24,0.7593,0.7226,168.9710,0.0,2565.4824,2667.2912,71.9046


,모형,변수,계수,군집 표준오차,t값,p-value
0,OLS_보행시간구간더미,2024년 종일 승차인원,0.7506,0.1531,4.9026,0.0000
1,OLS_보행시간구간더미,GTX 보행 20분 이내,-56.2864,31.5896,-1.7818,0.0748
2,OLS_보행시간구간더미,GTX 보행 20~60분,-9.6186,11.3303,-0.8489,0.3959
3,OLS_보행시간구간더미,IC까지 도로거리(km),-3.3555,2.3792,-1.4103,0.1584
4,OLS_보행시간구간더미,GTX 경쟁 목적지 비율(%p),-0.0699,0.0991,-0.7059,0.4803
5,OLS_보행시간구간더미,2024년 20~50대 인구(명),0.0127,0.0179,0.7099,0.4778
6,OLS_보행시간구간더미,2024년 건축물 수(동),0.0195,0.1760,0.1105,0.9120
7,OLS_보행시간연속형,2024년 종일 승차인원,0.7293,0.1614,4.5175,0.0000
8,OLS_보행시간연속형,GTX역 보행시간(분),0.1015,0.1714,0.5920,0.5538
9,OLS_보행시간연속형,IC까지 도로거리(km),-2.3381,2.5747,-0.9081,0.3638


## 해석 주의사항

이 모형은 2024년 승차규모가 동일한 셀×노선끼리 비교했을 때 각 지역 특성과 2025년 승차인원의 관계를 추정합니다. 셀×노선 승차인원은 비음수 정수이고 분포가 치우칠 수 있으므로, OLS는 NB2·Poisson QMLE의 비교 및 강건성 분석으로 사용합니다.

## 6. 변화량 OLS — 2024년 승차인원 통제 제외

종속변수를 `2025년 승차인원 - 2024년 승차인원`으로 설정하고 2024년 승차인원은 독립변수에서 제외합니다. 보행시간 구간 더미 모형과 연속형 모형을 각각 추정합니다.

In [7]:
def fit_change_ols(name, features):
    d = data.dropna(subset=['승차인원_증감'] + features).copy()
    route_fe = pd.get_dummies(d[ROUTE].astype(str), prefix='노선', drop_first=True, dtype=float)
    X = sm.add_constant(pd.concat([d[features].astype(float), route_fe], axis=1), has_constant='add')
    model = sm.OLS(d['승차인원_증감'].astype(float), X)
    classical = model.fit()
    clustered = model.fit(cov_type='cluster', cov_kwds={'groups':pd.factorize(d[ID])[0], 'use_correction':True})
    ci = clustered.conf_int().loc[features]
    result = pd.DataFrame({
        '변수':[LABELS[v] for v in features], '계수':[clustered.params[v] for v in features],
        '군집 표준오차':[clustered.bse[v] for v in features], 't값':[clustered.tvalues[v] for v in features],
        'p-value':[clustered.pvalues[v] for v in features],
        '95% CI 하한':[ci.loc[v,0] for v in features], '95% CI 상한':[ci.loc[v,1] for v in features]
    })
    info = {'모형':name, '표본 수':len(d), '격자 수':d[ID].nunique(), '노선 수':d[ROUTE].nunique(),
            'R²':classical.rsquared, '수정 R²':classical.rsquared_adj,
            '군집강건 F값':float(clustered.fvalue), '군집강건 F p값':float(clustered.f_pvalue),
            'AIC':classical.aic, 'BIC':classical.bic, 'RMSE':float(np.sqrt(np.mean(classical.resid**2)))}
    return {'name':name, 'data':d, 'X':X, 'classical':classical, 'fit':clustered, 'result':result, 'info':info}
CHANGE_DUMMY_FEATURES = ['walk_20', 'walk_20_60', 'ic_access', 'competing_share', 'pop_20_50', 'buildings']
CHANGE_CONT_FEATURES = ['walk_min', 'ic_access', 'competing_share', 'pop_20_50', 'buildings']
change_dummy = fit_change_ols('변화량OLS_보행시간구간더미_2024통제없음', CHANGE_DUMMY_FEATURES)
change_continuous = fit_change_ols('변화량OLS_보행시간연속형_2024통제없음', CHANGE_CONT_FEATURES)
display(pd.DataFrame([change_dummy['info'], change_continuous['info']]).round(4))
display(change_dummy['result'].round(4))
display(change_continuous['result'].round(4))


,모형,표본 수,격자 수,노선 수,R²,수정 R²,군집강건 F값,군집강건 F p값,AIC,BIC,RMSE
0,변화량OLS_보행시간구간더미_2024통제없음,220,132,24,0.1900,0.0663,130.4455,0.0,2588.6663,2690.4751,75.7948
1,변화량OLS_보행시간연속형_2024통제없음,220,132,24,0.1498,0.0252,161.3422,0.0,2597.3124,2695.7276,77.6511


,변수,계수,군집 표준오차,t값,p-value,95% CI 하한,95% CI 상한
0,GTX 보행 20분 이내,-94.8827,36.7269,-2.5835,0.0098,-166.8662,-22.8993
1,GTX 보행 20~60분,-27.3709,13.1555,-2.0806,0.0375,-53.1552,-1.5866
2,IC까지 도로거리(km),-3.6064,2.5696,-1.4035,0.1605,-8.6427,1.4299
3,GTX 경쟁 목적지 비율(%p),0.0060,0.1162,0.0515,0.9590,-0.2218,0.2338
4,2024년 20~50대 인구(명),0.0087,0.0197,0.4422,0.6583,-0.0298,0.0472
5,2024년 건축물 수(동),0.0976,0.2462,0.3962,0.6919,-0.3851,0.5802


,변수,계수,군집 표준오차,t값,p-value,95% CI 하한,95% CI 상한
0,GTX역 보행시간(분),0.4351,0.1909,2.2797,0.0226,0.0610,0.8092
1,IC까지 도로거리(km),-2.7907,2.8848,-0.9674,0.3333,-8.4447,2.8633
2,GTX 경쟁 목적지 비율(%p),0.0374,0.1260,0.2972,0.7663,-0.2095,0.2844
3,2024년 20~50대 인구(명),0.0106,0.0212,0.5010,0.6164,-0.0309,0.0521
4,2024년 건축물 수(동),-0.0245,0.2113,-0.1161,0.9076,-0.4387,0.3896


## 7. 2024년 승차규모 통제 여부 비교

In [8]:
control_comparison = pd.DataFrame([
    {'구분':'2024 승차 통제', **ols_dummy['info']},
    {'구분':'2024 승차 통제 없음', **change_dummy['info']}
])
display(control_comparison.round(4))
coef_comparison = ols_dummy['result'][['변수','계수','p-value']].merge(
    change_dummy['result'][['변수','계수','p-value']], on='변수', how='outer', suffixes=('_통제', '_통제없음'))
display(coef_comparison.round(4))


,구분,모형,표본 수,격자 수,노선 수,R²,수정 R²,군집강건 F값,군집강건 F p값,AIC,BIC,RMSE
0,2024 승차 통제,OLS_보행시간구간더미,220,132,24,0.7664,0.7293,93.4178,0.0,2560.9584,2666.1608,70.8463
1,2024 승차 통제 없음,변화량OLS_보행시간구간더미_2024통제없음,220,132,24,0.1900,0.0663,130.4455,0.0,2588.6663,2690.4751,75.7948


,변수,계수_통제,p-value_통제,계수_통제없음,p-value_통제없음
0,2024년 20~50대 인구(명),0.0127,0.4778,0.0087,0.6583
1,2024년 건축물 수(동),0.0195,0.9120,0.0976,0.6919
2,2024년 종일 승차인원,0.7506,0.0000,NaN,NaN
3,GTX 경쟁 목적지 비율(%p),-0.0699,0.4803,0.0060,0.9590
4,GTX 보행 20~60분,-9.6186,0.3959,-27.3709,0.0375
5,GTX 보행 20분 이내,-56.2864,0.0748,-94.8827,0.0098
6,IC까지 도로거리(km),-3.3555,0.1584,-3.6064,0.1605
